# 03 — Feature Engineering

**Purpose:** Compute all features needed for modeling:
1. Map 18 shot types → binary offensive/defensive label
2. Compute court-center distance per stroke
3. Define easy/hard binary split on distance (median split)
4. Compute inter-shot interval (reaction time) per stroke
5. Filter implausible reaction times (RT < 100ms or RT > 3s)

**Input:** `EDA/data/shuttleset_filtered_players.parquet`

**Output:** `EDA/data/shuttleset_features.parquet`

## Imports and load

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("data")

input_path = OUTPUT_DIR / "shuttleset_filtered_players.parquet"
assert input_path.exists(), (
    f"Input not found: {input_path}\n"
)

df = pd.read_parquet(input_path)
print(f"Loaded: {len(df):,} strokes, {df['player_name'].nunique()} players")
print(f"Columns: {df.columns.tolist()}")

Loaded: 25,747 strokes, 22 players
Columns: ['rally', 'ball_round', 'time', 'frame_num', 'roundscore_A', 'roundscore_B', 'player', 'server', 'type', 'aroundhead', 'backhand', 'hit_height', 'hit_area', 'hit_x', 'hit_y', 'landing_height', 'landing_area', 'landing_x', 'landing_y', 'lose_reason', 'win_reason', 'getpoint_player', 'flaw', 'player_location_area', 'player_location_x', 'player_location_y', 'opponent_location_area', 'opponent_location_x', 'opponent_location_y', 'db', 'match_id', 'set_num', 'stroke_id', 'type_en', 'player_name', 'opponent_name']


## 1. Map shot types → offensive/defensive binary label

Each of the 18 shot types is classified as offensive (1) or defensive (0)

In [2]:
SHOT_OFFENSIVE_MAP = {
    # Offensive shots
    "smash":                  1,
    "wrist smash":            1,
    "drop":                   1,
    "net shot":               1,
    "rush":                   1,
    "push":                   1,
    "drive":                  1,
    "driven flight":          1,
    "back-court drive":       1,
    "cross-court net shot":   1,
    # Defensive shots
    "clear":                  0,
    "lob":                    0,
    "return net":             0,
    "defensive return lob":   0,
    "defensive return drive": 0,
    "passive drop":           0,
    # Services — neutral, excluded from offensive label
    "short service":          np.nan,
    "long service":           np.nan,
}

df["is_offensive"] = df["type_en"].map(SHOT_OFFENSIVE_MAP)

print("Offensive label distribution:")
print(df["is_offensive"].value_counts(dropna=False))
print(f"\nService strokes (excluded, NaN): {df['is_offensive'].isna().sum():,}")

Offensive label distribution:
is_offensive
1.0    13477
0.0     8873
NaN     3397
Name: count, dtype: int64

Service strokes (excluded, NaN): 3,397


## 2. Compute court-center distance per stroke

Distance = √((x − x̄)² + (y − ȳ)²) where x̄, ȳ are the grand mean
of all player positions across the entire dataset.

In [3]:
x_mean = df["player_location_x"].mean()
y_mean = df["player_location_y"].mean()

print(f"Grand mean player position — x̄: {x_mean:.2f}, ȳ: {y_mean:.2f}")

df["court_center_distance"] = np.sqrt(
    (df["player_location_x"] - x_mean) ** 2
    + (df["player_location_y"] - y_mean) ** 2
)

print(f"\ncourt_center_distance summary:")
print(df["court_center_distance"].describe().round(2))

Grand mean player position — x̄: 636.25, ȳ: 450.64

court_center_distance summary:
count    25747.00
mean       137.34
std         51.24
min          1.30
25%        100.53
50%        133.58
75%        166.82
max        459.39
Name: court_center_distance, dtype: float64


## 3. Easy/hard binary split on distance (median split)

Strokes where the player is closer to the court center than the median
are classified as easy (0). Strokes farther than the median are hard (1).

In [4]:
distance_median = df["court_center_distance"].median()
print(f"Distance median threshold: {distance_median:.4f}")
print("Strokes closer than median → easy (0)")
print("Strokes farther than median → hard (1)")

df["is_hard"] = (df["court_center_distance"] > distance_median).astype(int)

print(f"\nEasy/hard split:")
print(df["is_hard"].value_counts())

Distance median threshold: 133.5801
Strokes closer than median → easy (0)
Strokes farther than median → hard (1)

Easy/hard split:
is_hard
0    12874
1    12873
Name: count, dtype: int64


## 4. Compute inter-shot interval (reaction time)

Reaction time (RT) for each stroke = time of current stroke minus time
of the opponent's previous stroke within the same rally.

RT is only meaningful for non-service strokes (ball_round > 1).
The `frame_num` column is used since it is a monotonic integer counter
derived from the video frame rate (frame_num = time_seconds × fps).
RT in seconds = (frame_num_current − frame_num_previous) / fps.

FPS is 30 based on the ShuttleSet paper.

In [5]:
FPS = 30

# Sort within each rally so shift is meaningful
df = df.sort_values(
    ["match_id", "set_num", "rally", "ball_round"]
).reset_index(drop=True)

# Previous frame number within the same rally
df["prev_frame_num"] = df.groupby(
    ["match_id", "set_num", "rally"]
)["frame_num"].shift(1)

# RT in seconds
df["reaction_time_s"] = (
    df["frame_num"] - df["prev_frame_num"]
) / FPS

# First stroke of each rally has no previous stroke — RT is NaN by design
n_valid = df["reaction_time_s"].notna().sum()
print(f"Strokes with valid RT: {n_valid:,}")
print(f"First strokes (NaN RT): {df['reaction_time_s'].isna().sum():,}")
print(f"\nRT summary (seconds):")
print(df["reaction_time_s"].describe().round(3))

Strokes with valid RT: 23,281
First strokes (NaN RT): 2,466

RT summary (seconds):
count    23281.000
mean         0.831
std          0.388
min        -16.700
25%          0.633
50%          0.800
75%          1.000
max         18.933
Name: reaction_time_s, dtype: float64


## 5. Filter implausible reaction times

Drop strokes where RT < 100ms or RT > 3s.
These are likely annotation errors or rally boundary artifacts.

In [6]:
RT_MIN = 0.1
RT_MAX = 3.0

# Strokes where RT exists but is outside the plausible range
implausible_mask = (
    df["reaction_time_s"].notna()
    & (
        (df["reaction_time_s"] < RT_MIN)
        | (df["reaction_time_s"] > RT_MAX)
    )
)

n_implausible = implausible_mask.sum()
print(f"Implausible RT strokes dropped: {n_implausible:,}")
print(f"  RT < {RT_MIN}s: {(df['reaction_time_s'] < RT_MIN).sum():,}")
print(f"  RT > {RT_MAX}s: {(df['reaction_time_s'] > RT_MAX).sum():,}")

df = df[~implausible_mask].reset_index(drop=True)

print(f"\nStrokes remaining: {len(df):,}")
print(f"\nRT summary after filtering (seconds):")
print(df["reaction_time_s"].describe().round(3))

Implausible RT strokes dropped: 52
  RT < 0.1s: 50
  RT > 3.0s: 2

Strokes remaining: 25,695

RT summary after filtering (seconds):
count    23229.000
mean         0.834
std          0.322
min          0.100
25%          0.633
50%          0.800
75%          1.000
max          2.933
Name: reaction_time_s, dtype: float64


## 6. Save output

In [7]:
output_path = OUTPUT_DIR / "shuttleset_final_features.parquet"
df.to_parquet(output_path, index=False)

print(f"Saved: {output_path}")
print(f"\nFinal shape: {df.shape}")
print(f"New columns added:")
new_cols = [
    "is_offensive",
    "court_center_distance",
    "is_hard",
    "prev_frame_num",
    "reaction_time_s",
]
print(df[new_cols].describe().round(3))

Saved: data/shuttleset_final_features.parquet

Final shape: (25695, 41)
New columns added:
       is_offensive  court_center_distance  is_hard  prev_frame_num  \
count     22299.000              25695.000  25695.0       23229.000   
mean          0.604                137.364      0.5       57252.219   
std           0.489                 51.253      0.5       30741.937   
min           0.000                  1.300      0.0        1874.000   
25%           0.000                100.590      0.0       31649.000   
50%           1.000                133.586      1.0       54741.000   
75%           1.000                166.827      1.0       79154.000   
max           1.000                459.389      1.0      157499.000   

       reaction_time_s  
count        23229.000  
mean             0.834  
std              0.322  
min              0.100  
25%              0.633  
50%              0.800  
75%              1.000  
max              2.933  
